# M3: MuRIL shared encoder + two heads on Kaggle (multi-task, tone-only, type-only)

This notebook runs the three M3 runs from Phase 3 on a Kaggle GPU. All the training
code lives in `scripts/train_multitask.py`; the notebook only calls it with
Kaggle's file paths.

**Checkpoints.** After each seed finishes, its scores and predictions are saved. If the
session stops halfway, run all cells again: finished seeds are skipped.
(The ~950 MB model weights are *not* saved, to keep the download small.)

## Setup on Kaggle
1. **Update your private dataset.** Open the dataset you made for M2 → *New Version* →
   upload `tulu-hope-kaggle-dataset.zip` (it now also contains `train_multitask.py`) →
   *Create*. No test file goes in it.
2. **Import this notebook.** From the dataset page, click *New Notebook* (this attaches the
   dataset automatically) → *File* → *Import Notebook* → `notebooks/m3_muril_kaggle.ipynb`.
   Check that the dataset is listed under *Input* in the right-hand panel.
3. **Session options** (right-hand panel):
   - *Accelerator* → **GPU T4 x2**. (Avoid P100: newer PyTorch versions on Kaggle no longer support it.)
   - *Internet* → **On**. MuRIL is downloaded from Hugging Face the first time it is used.
     If you don't see the Internet option, your account needs phone verification (already done).
4. **Quick test first.** Set `QUICK_TEST = True`, run every cell (about 3–5 minutes).
   Then set it back to `False` → *Save Version* → *Save & Run All (Commit)*.

A warning like *"You are sending unauthenticated requests to the HF Hub"* is harmless.

## Bringing the results back to the Mac
When the version is *Complete*: *Output* tab → download `m3_muril_outputs.zip` →
unzip it, and tell Claude where the unzipped folder is.

In [ ]:
# Step 1: find the input files and choose where outputs go.
from pathlib import Path
import torch

ON_KAGGLE = Path("/kaggle/input").exists()

# Set to True for a quick check (1 seed, 1 epoch). Set to False for the real run.
QUICK_TEST = False

if ON_KAGGLE:
    def find(name):
        # rglob searches every sub-folder of /kaggle/input for a file with this name.
        matches = sorted(Path("/kaggle/input").rglob(name))
        if not matches:
            raise FileNotFoundError(f"{name} not found. Did you attach the NEW version of the dataset?")
        return matches[0]

    SCRIPT = find("train_multitask.py")
    TRAIN = find("train_clean.csv")
    DEV = find("dev_multitask.csv")
    OUT = Path("/kaggle/working/m3_outputs")   # /kaggle/working is the folder Kaggle keeps
else:
    SCRIPT = Path("../scripts/train_multitask.py")
    TRAIN = Path("../data/train_clean.csv")
    DEV = Path("../data/dev_multitask.csv")
    OUT = Path("..")

RESULTS = OUT / "results"
CKPT = OUT / "checkpoints"
LOGS = RESULTS / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

NAME = "m3_muril-base-cased"   # matches the file names the script writes
EXTRA = "--seeds 1 --epochs 1" if QUICK_TEST else ""

print("script :", SCRIPT)
print("train  :", TRAIN)
print("dev    :", DEV)
print("outputs:", OUT)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")

In [ ]:
# Step 1b: check that MuRIL can be downloaded (needs Internet = On).
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("google/muril-base-cased")
print("MuRIL tokenizer OK. Example:", tok.tokenize("baari porlu movie"))

## Step 2: the three runs
Each cell runs `train_multitask.py` once (5 seeds × 5 epochs). A line starting with `!`
is a shell command; `{SCRIPT}` etc. insert the Python variables from Step 1.

In [ ]:
# multi-task: shared MuRIL encoder, both heads trained
!python -u {SCRIPT} --task multi --train {TRAIN} --dev {DEV} --out_dir {RESULTS} --ckpt_dir {CKPT} {EXTRA} 2>&1 | tee {LOGS}/{NAME}_multi.log

In [ ]:
# single-task control: only the tone head is trained
!python -u {SCRIPT} --task coarse --train {TRAIN} --dev {DEV} --out_dir {RESULTS} --ckpt_dir {CKPT} {EXTRA} 2>&1 | tee {LOGS}/{NAME}_coarse.log

In [ ]:
# single-task control: only the hope-type head is trained
!python -u {SCRIPT} --task fine --train {TRAIN} --dev {DEV} --out_dir {RESULTS} --ckpt_dir {CKPT} {EXTRA} 2>&1 | tee {LOGS}/{NAME}_fine.log

In [ ]:
# Step 3: summary of the three runs, and a check for classes stuck at 0.000 F1.
import json
import pandas as pd

rows = []
for task in ["multi", "coarse", "fine"]:
    path = RESULTS / f"{NAME}_{task}.json"
    if not path.exists():
        print("missing:", path)
        continue
    r = json.loads(path.read_text())
    rows.append({"task": task,
                 "tone F1": r["tone_mean"], "tone std": r["tone_std"],
                 "type F1": r["type_mean"], "type std": r["type_std"],
                 "best epochs": r["best_epochs"]})

    for i, seed_scores in enumerate(r["per_class_f1_by_seed"]):
        for head, classes in seed_scores.items():
            for name, f1 in classes.items():
                if f1 == 0.0:
                    print(f"WARNING {task}: seed {r['seeds'][i]} has 0.000 F1 for {head} class '{name}'")

print("M1 baseline to beat: tone 0.579, type 0.400")
print("Best M2 (CNN):       tone 0.558, type 0.411")
pd.DataFrame(rows).round(3)

In [ ]:
# Step 4 (Kaggle only): pack results + checkpoints into one zip for the Output tab.
import shutil

if ON_KAGGLE:
    zip_path = shutil.make_archive("/kaggle/working/m3_muril_outputs", "zip", root_dir=OUT)
    print("created", zip_path, "- download it from the Output tab")
else:
    print("Running locally: results are already in the project's results/ folder.")